# 06C GROMACS Solvated System

This notebook is the main realistic GROMACS production-preparation workflow for explicit solvent PHA oligomer simulations.

It starts from the dry GROMACS polymer inputs prepared in 06B, adds a periodic box, TIP3P-compatible water, SOD/CLA ions, PME-ready preprocessing, and engine-specific NVT/NPT/production scripts. It does not submit HPC jobs.


## Workflow Scope

- Engine: GROMACS
- System: explicit solvent polymer
- Inputs: fresh `gromacs/dry_polymer/topol.top` and `step5_input.gro`
- Output: `gromacs/solvated_polymer/`
- Core GROMACS steps: `editconf`, `make_ndx`, `solvate`, `grompp`, `genion`, final `editconf`, final `make_ndx`, minimisation `grompp`

Do not rerun `gmx solvate` or `gmx genion` manually on an already modified topology. Use this notebook or the generated clean script so the workflow resets from the dry polymer topology first.


In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
system_name = "PHB4"
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gromacs_dir = md_root / "gromacs"
dry_polymer_dir = gromacs_dir / "dry_polymer"
solvated_polymer_dir = gromacs_dir / "solvated_polymer"

required_inputs = [
    dry_polymer_dir / "topol.top",
    dry_polymer_dir / "step5_input.gro",
    dry_polymer_dir / "index.ndx",
]

for path in required_inputs:
    print(f"{path.name}: {path.exists()} -> {path}")

Repository: /Users/k20098771/opt/iPHASimulator_v2
System: PHB4
MD output root: /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4
topol.top: True -> /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/dry_polymer/topol.top
step5_input.gro: True -> /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/dry_polymer/step5_input.gro
index.ndx: True -> /Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/dry_polymer/index.ndx


## Write Solvation Workflow Files

The generated `run_solvate_local.sh` is reset-safe. It copies fresh dry polymer inputs, removes stale generated files, writes separate logs for each GROMACS command, and standardises the final solvated/ionised starting structure as `step5_input.gro`.


In [2]:
import importlib
import iphasimulator.simulation.gromacs_runner as gromacs_runner

gromacs_runner = importlib.reload(gromacs_runner)
write_gromacs_solvation_files = gromacs_runner.write_gromacs_solvation_files

solvation_files = write_gromacs_solvation_files(
    gromacs_dir,
    workflow_type="polymer",
    box_padding_nm=1.2,
    ion_concentration_molar=0.15,
    clean=True,
)

solvation_files

GromacsSolvationFiles(output_dir=PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer'), ions_mdp_path=PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/ions.mdp'), solvation_itp_paths=(PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/ions.mdp'), PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/tip3_ions_atomtypes.itp'), PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/TIP3_SOL.itp'), PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/TIP3.itp'), PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/SOD.itp'), PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/POT.itp

## Run Local Solvation When Ready

This step writes water, ions, final coordinates, index files, and `minim_grompp.log`. It is disabled by default. The default solvent group is `SOL`; if your `make_ndx` output uses a different name, run the script with `--solvent-group <name>`.


In [3]:
import subprocess

RUN_GROMACS_SOLVATION = True

if RUN_GROMACS_SOLVATION:
    result = subprocess.run(
        ["bash", "run_solvate_local.sh"],
        cwd=solvated_polymer_dir,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Set RUN_GROMACS_SOLVATION = True to run GROMACS solvation and ion insertion.")

Reset topol.top and step5_input.gro from ../dry_polymer.
editconf_box completed; log: editconf_box.log
make_ndx_box completed; log: make_ndx_box.log
solvate completed; log: solvate.log
Inspecting topol.top after gmx solvate for water and ion topology definitions.
Added tip3_ions_atomtypes.itp include to topol.top
Added TIP3_SOL.itp include to topol.top
Added SOD.itp include to topol.top
Added CLA.itp include to topol.top
topol.top contains required SOL/SOD/CLA molecule definitions and OT/HT/SOD/CLA atom types.
ions_grompp completed; log: ions_grompp.log
genion completed; log: genion.log
editconf_final completed; log: editconf_final.log
make_ndx_final completed; log: make_ndx_final.log
Final step5_input.gro atom count: 7394
Final topol.top [ molecules ] section:
  PHA 1
  SOL 2443
  SOD 7
  CLA 7
minim_grompp completed; log: minim_grompp.log



## Validate Solvated Topology and Coordinates

The final `step5_input.gro` must match `topol.top`. For PHB4 with 0.15 M salt, expect SOD and CLA molecule counts after the solvation script has run.


In [4]:
from iphasimulator.simulation.gromacs_runner import (
    validate_gromacs_coordinate_topology_counts,
    validate_gromacs_solvated_topology,
)

if (solvated_polymer_dir / "step5_input.gro").exists():
    count_validation = validate_gromacs_coordinate_topology_counts(solvated_polymer_dir)
    topology_validation = validate_gromacs_solvated_topology(solvated_polymer_dir)
    print(f"Coordinate atoms: {count_validation.coordinate_atom_count}")
    print(f"Expected topology atoms: {count_validation.expected_atom_count}")
    print(f"Counts match: {count_validation.valid}")
    print(topology_validation.molecule_counts)
else:
    print(f"Missing final solvated coordinates: {solvated_polymer_dir / 'step5_input.gro'}")

Coordinate atoms: 7394
Expected topology atoms: 7394
Counts match: True
{'PHA': 1, 'SOL': 2443, 'SOD': 7, 'CLA': 7}


## Production Preparation

This notebook prepares local GROMACS minimisation and engine-specific NVT/NPT/production scripts. Submit or benchmark those scripts from notebook 07 only.


## Generated Polymer Run Stages

The generated GROMACS polymer scripts use the following stages after solvation and ion insertion. The conceptual comparison between workflow branches is described at the end of notebook 05.

| Stage | What it does | Simulation length | Why it is included |
| --- | --- | ---: | --- |
| `step6.0_minimization` | Energy minimisation of the solvated/ionised box | Not time-based | Removes close contacts before molecular dynamics starts |
| `step6.1_nvt` | Constant-volume temperature equilibration at 300 K | 100 ps | Assigns/generates velocities and stabilises temperature without changing box volume |
| `step6.2_npt` | Constant-pressure density equilibration at 300 K and 1 bar | 500 ps | Lets the solvent box density and pressure relax before production |
| `step7_production` | Production MD at 300 K and 1 bar | 100 ns by default | Generates the trajectory used by preprocessing and analysis notebooks |

The physical simulation length comes from each `.mdp` file as `nsteps * dt`. The current polymer templates use `dt = 0.002 ps`: `step6.1_nvt.mdp` is 50,000 steps, `step6.2_npt.mdp` is 250,000 steps, and `step7_production.mdp` is 50,000,000 steps.

Wall-clock runtime is hardware-dependent. Use this workflow to iterate quickly and to benchmark polymer-only systems before spending time on more conservative protocols.


In [5]:
{
    "local_minimization_script": solvated_polymer_dir / "run_step6_local.sh",
    "hpc_script_for_07": solvated_polymer_dir / "run_hpc_equilibration_production.slurm",
    "final_start_structure": solvated_polymer_dir / "step5_input.gro",
    "final_topology": solvated_polymer_dir / "topol.top",
}

{'local_minimization_script': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/run_step6_local.sh'),
 'hpc_script_for_07': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/run_hpc_equilibration_production.slurm'),
 'final_start_structure': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/step5_input.gro'),
 'final_topology': PosixPath('/Users/k20098771/opt/iPHASimulator_v2/examples/output/md_tests/PHB4/gromacs/solvated_polymer/topol.top')}